In [1]:

import mlflow
# Step 2: Set up the MLflow tracking server
mlflow.set_tracking_uri("http://ec2-16-170-217-141.eu-north-1.compute.amazonaws.com:5000/")

In [2]:
# set or create an experiment
mlflow.set_experiment("exp_5 ml_algo_with_hp_tunning")


<Experiment: artifact_location='s3://my-s3-bucket-of-store-artifact-youtube-data12/mlflow-artifacts/7', creation_time=1764058549473, experiment_id='7', last_update_time=1764058549473, lifecycle_stage='active', name='exp_5 ml_algo_with_hp_tunning', tags={}>

In [11]:
import pandas as pd
df=pd.read_csv('sentiment_clean.csv')

In [12]:
df['sentiment_numeric']=df.pop('sentiment_numeric')

In [13]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
import joblib
import mlflow
import optuna

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, recall_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.utils.class_weight import compute_class_weight
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
import seaborn as sns

# -------------------------
# Target Fix 
# -------------------------
df['sentiment_numeric'] = df['sentiment_numeric'].map({-1: 2, 0: 0, 1: 1})


# -------------------------
# Numeric + Text Split
# -------------------------
X_numeric = df.iloc[:, 1:-1]
y = df['sentiment_numeric']

scaler = StandardScaler(with_mean=False)
X_numeric_scaled = scaler.fit_transform(X_numeric)

X_train_num, X_test_num, y_train, y_test, train_idx, test_idx = train_test_split(
    X_numeric_scaled, y, df.index,
    test_size=0.20, random_state=42, stratify=y
)

tfidf = TfidfVectorizer(ngram_range=(1,3), max_features=20000)
X_train_text = tfidf.fit_transform(df.loc[train_idx, 'text_clean'])
X_test_text = tfidf.transform(df.loc[test_idx, 'text_clean'])

X_train = sp.hstack([X_train_text, sp.csr_matrix(X_train_num)])
X_test = sp.hstack([X_test_text, sp.csr_matrix(X_test_num)])


# -------------------------
# Class Weights
# -------------------------
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = {cls: w for cls, w in zip(np.unique(y_train), class_weights)}

print("Class Weights:", class_weight_dict)


# -------------------------
# OPTUNA Tuning Function
# -------------------------
def objective(trial):

    C = trial.suggest_float("C", 0.1, 5.0, log=True)
    solver = trial.suggest_categorical("solver", ["liblinear", "lbfgs"])

    model = LogisticRegression(
        C=C,
        solver=solver,
        max_iter=1500,
        class_weight=class_weight_dict
    )

    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    return recall_score(y_test, preds, average="macro")  # main metric


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=15)

print("\nBest Hyperparameters:", study.best_params)


# -------------------------
# Train Final Model with Best Params
# -------------------------
best_params = study.best_params

model = LogisticRegression(
    C=best_params["C"],
    solver=best_params["solver"],
    max_iter=1500,
    class_weight=class_weight_dict
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)


# -------------------------
# Evaluation
# -------------------------
print("\n=================== FINAL RESULTS ===================")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:\n", cm)


# -------------------------
# MLflow Logging
# -------------------------
with mlflow.start_run(run_name="LR_Optuna_Tuned"):

    mlflow.log_param("model", "Logistic Regression + Optuna")
    mlflow.log_params(best_params)
    mlflow.log_param("ngram_range", "(1,3)")
    mlflow.log_param("max_features", 20000)
    mlflow.log_param("class_weight", str(class_weight_dict))

    mlflow.log_metric("accuracy", accuracy_score(y_test, y_pred))
    mlflow.log_metric("macro_recall", recall_score(y_test, y_pred, average="macro"))

    plt.figure(figsize=(7,5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title("Confusion Matrix")
    plt.savefig("cm_lr.png")
    mlflow.log_artifact("cm_lr.png")
    plt.close()

    joblib.dump(model, "logistic_regression_optuna.pkl")
    mlflow.log_artifact("logistic_regression_optuna.pkl")

print("\nModel Saved Successfully with Optuna Tuning! 🚀")


[I 2025-11-29 18:48:26,477] A new study created in memory with name: no-name-cf1ae2d8-9e9b-4335-9c66-8d3fcb8351d4


Class Weights: {np.int64(0): np.float64(0.7256444102225644), np.int64(1): np.float64(0.7350641632774342), np.int64(2): np.float64(3.8242512077294686)}


[I 2025-11-29 18:48:34,088] Trial 0 finished with value: 0.7303748620301893 and parameters: {'C': 0.20369668634388172, 'solver': 'lbfgs'}. Best is trial 0 with value: 0.7303748620301893.
[I 2025-11-29 18:48:53,522] Trial 1 finished with value: 0.7282098637282917 and parameters: {'C': 1.7783468536865397, 'solver': 'lbfgs'}. Best is trial 0 with value: 0.7303748620301893.
e:\mlops_projects\influence_mirror\myenv\Lib\site-packages\sklearn\linear_model\_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(
[I 2025-11-29 18:48:57,026] Trial 2 finished with value: 0.705926858086888 and parameters: {'C': 0.11801861972813953, 'solver': 'liblinear'}. Best is trial 0 with value: 0.7303748620301893.
[I 2025-11-29 18:49:25,130] Trial 3 finished wit


Best Hyperparameters: {'C': 0.5776154649566887, 'solver': 'lbfgs'}

=================== FINAL RESULTS ===================
Accuracy: 0.7207962813257882

Classification Report:

              precision    recall  f1-score   support

           0       0.72      0.74      0.73      4546
           1       0.81      0.69      0.74      4488
           2       0.49      0.78      0.60       862

    accuracy                           0.72      9896
   macro avg       0.67      0.74      0.69      9896
weighted avg       0.74      0.72      0.72      9896


Confusion Matrix:
 [[3377  667  502]
 [1211 3083  194]
 [ 121   68  673]]
🏃 View run LR_Optuna_Tuned at: http://ec2-16-170-217-141.eu-north-1.compute.amazonaws.com:5000/#/experiments/7/runs/14d2014d465243dd871afd99dec92b50
🧪 View experiment at: http://ec2-16-170-217-141.eu-north-1.compute.amazonaws.com:5000/#/experiments/7

Model Saved Successfully with Optuna Tuning! 🚀
